# 쌤쌤 세션 3 — FootMR Colab 노트북

영상 → 3D 모션(SMPL). GVHMR을 따로 설치할 필요 없음 — FootMR 저장소가 GVHMR 파이프라인 전체를 포함하고
그 위에 발 보정까지 해준다(`tools/demo.py` 하나로 끝). 근거: `samsam_dev_spec.md`, `samsam_plan.md` 세션 3.

**위에서부터 순서대로 실행할 것.** 2026-09-11에 처음부터 끝까지 한 번 밟으며 함정을 전부 주석으로 박아뒀다.
특히 셀 4의 설치 순서는 바꾸지 말 것 — pip은 목록 중 하나만 실패해도 **한 패키지도 안 깔고** 중단하기
때문에, 순서가 어긋나면 엉뚱한 곳에서 `ModuleNotFoundError`가 나고 원인을 못 찾는다.

## 시작 전 체크리스트
- [ ] 런타임 유형 = GPU(T4). 상단 메뉴 런타임 > 런타임 유형 변경
- [ ] SMPL(smpl.is.tue.mpg.de "for Python users")·SMPL-X(smpl-x.is.tue.mpg.de) 가입 + 다운로드
      — 라이선스 동의가 걸려 있어 URL로 못 받는다. 한 번 받아 드라이브에 넣어두면 그다음부터 자동.
- [ ] 입력 영상 준비 (고정 카메라 — `samsam_shooting_guide.md` 기준)

전체 소요: 체크포인트가 드라이브에 캐시돼 있으면 설치 ~10분 + 추론 몇 분.

In [ ]:
# 1. GPU 확인 — T4가 보여야 정상. CPU로 잡히면 여기서 런타임 유형부터 바꿀 것.
!nvidia-smi

In [ ]:
# 2. Drive 마운트 — 체크포인트 6GB를 매 세션 다시 받지 않으려면 필수.
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_CACHE = '/content/drive/MyDrive/samsam_footmr_checkpoints'
ASSETS_FOLDER_ID = '1-I1xUrX6nlo-sfUmHlWlGp_syazt_rPu'   # 내 드라이브: body_models + footmr/vitpose 체크포인트
HMR2_FILE_ID     = '1GtCB5cfetGC8mfIiMFLCBeGRt0iTqJZr'   # hmr2 ckpt를 내 드라이브로 "사본 만들기" 한 것
os.makedirs(CKPT_CACHE, exist_ok=True)
print('체크포인트 캐시:', CKPT_CACHE)

In [ ]:
# 3. FootMR clone. 이후 모든 셀은 /content/FootMR 기준 상대경로를 쓴다.
!git clone --recursive --depth 1 --shallow-submodules https://github.com/twehrbein/FootMR.git
%cd /content/FootMR

In [ ]:
# 4. python3.10 + 의존성. 2026-09-11에 실제로 밟은 함정 4개를 순서에 반영했다. 순서를 바꾸지 말 것.
%cd /content/FootMR
!apt-get update -qq
!apt-get install -y python3.10 python3.10-distutils python3.10-tk > /dev/null
# python3.10-tk: FootMR body_model.py 첫 줄의 죽은 `from turtle import forward`가 tkinter를 요구한다.
# python3.10 고정 이유: chumpy가 `inspect.getargspec`을 쓰는데 3.11에서 제거됐다. 3.12로 우회 불가.
!curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py
!python3.10 /tmp/get-pip.py

# ── 함정 1. numpy ────────────────────────────────────────────────────────────
# 데비안이 apt로 깐 numpy는 RECORD 파일이 없어 pip이 제거하지 못하고(`uninstall-no-record-file`)
# 설치 전체를 롤백한다. 게다가 그 numpy는 python3.12용 빌드라 3.10에서는 C 확장이 안 보인다
# (`No module named numpy.core._multiarray_umath`). --ignore-installed 로 덮어쓰면
# /usr/local/lib/python3.10/dist-packages 가 sys.path에서 먼저라 이긴다.
!python3.10 -m pip install --ignore-installed --no-cache-dir numpy==1.23.5 "setuptools<81" wheel

# ── 함정 2. chumpy ───────────────────────────────────────────────────────────
# setup.py가 `from pip._internal.req import parse_requirements`로 pip 비공개 내부 API를 임포트한다.
# 현대 pip에선 그 경로가 사라져 빌드가 구조적으로 불가능하고, pip은 이유도 `No available output.`으로만
# 남긴다. sdist에 C 확장이 0개인 순수 파이썬 패키지라 복사하면 끝. (SMPL .pkl 안에 chumpy 배열이
# 들어 있어 smplx가 언피클할 때 반드시 필요 — 생략 불가.)
!python3.10 -m pip install "scipy>=0.13.0"
!curl -sL https://files.pythonhosted.org/packages/01/f7/865755c8bdb837841938de622e6c8b5cb6b1c933bde3bd3332f0cd4574f1/chumpy-0.70.tar.gz -o /tmp/chumpy.tar.gz
!tar xzf /tmp/chumpy.tar.gz -C /tmp
!cp -r /tmp/chumpy-0.70/chumpy /usr/local/lib/python3.10/dist-packages/

# ── 함정 3. 빌드가 깨지는데 실제로는 안 쓰는 패키지 ──────────────────────────
# cython_bbox, lapx: FootMR 저장소 전체에서 requirements.txt 말고는 등장하지 않는다(ByteTrack 흔적).
# 반대로 pytorch3d는 demo.py 8행이 직접 임포트하므로 절대 빼면 안 된다(미리 빌드된 휠이라 컴파일 없음).
!grep -vE "chumpy|cython_bbox|lapx|^torch==|^torchvision==" requirements.txt > /tmp/req.txt
!python3.10 -m pip install --ignore-installed -r /tmp/req.txt

# ── 함정 4. torch 버전 ───────────────────────────────────────────────────────
# 위에서 핀을 뺐으므로 lightning이 최신 torch를 끌어온다. 그러면 pytorch3d(휠 이름이
# py310_cu121_pyt230 = torch 2.3.0 ABI 전용)와 torchvision 0.18이 깨진다(`torch.library has no
# attribute register_fake`). 그래서 마지막에 짝 맞는 버전으로 못박는다. 또 --ignore-installed 로
# 덮어쓴 트리가 남으면 `pip list`는 0.18인데 파일은 0.19인 유령 상태가 되므로 디렉터리부터 지운다.
!rm -rf /usr/local/lib/python3.10/dist-packages/torch /usr/local/lib/python3.10/dist-packages/torch-*.dist-info
!rm -rf /usr/local/lib/python3.10/dist-packages/torchvision /usr/local/lib/python3.10/dist-packages/torchvision-*.dist-info
!python3.10 -m pip install --no-cache-dir torch==2.3.0+cu121 torchvision==0.18.0+cu121 \
    --extra-index-url https://download.pytorch.org/whl/cu121
!python3.10 -m pip install -e .

In [ ]:
# 5. 설치 검증 게이트 — demo는 한참 돌린 뒤에야 임포트 에러를 내므로 여기서 먼저 거른다.
!python3.10 -c "import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())"
# 기대: 2.3.0+cu121 0.18.0+cu121 True    ← False면 런타임이 GPU가 아니다

# 버전 문자열은 거짓말을 할 수 있다(함정 4). 실제 파일이 0.18인지는 이걸로 본다 — 0이어야 정상.
!grep -c register_fake /usr/local/lib/python3.10/dist-packages/torchvision/_meta_registrations.py

!python3.10 -c "import hydra, einops, chumpy, smplx, cv2, av, trimesh, lightning, timm, skimage, ultralytics, pycolmap, wis3d; from pytorch3d.transforms import quaternion_to_matrix; print('imports ok')"

## 체크포인트

`docs/INSTALL.md`가 요구하는 최종 구조:

```
inputs/checkpoints/
├── body_models/smpl/SMPL_{FEMALE,MALE,NEUTRAL}.pkl   # 가입 후 직접 다운로드 (237M x3)
├── body_models/smplx/SMPLX_{FEMALE,MALE,NEUTRAL}.npz # 가입 후 직접 다운로드 (104M x3)
├── footmr/footmr_checkpoint.ckpt                      # 178,872,284 B
├── hmr2/epoch=10-step=25000.ckpt                      # 2,709,494,041 B
├── vitpose/vitpose-h-wholebody.pth                    # 2,549,087,447 B
└── yolo/yolov8x.pt                                    # 136,890,692 B
```

`dpvo/dpvo.pth`는 SLAM용이라 `-s`(정적 카메라)에서는 안 쓴다. `vitpose-h-multi-coco.pth`도
FootMR 경로에서는 wholebody만 로드하므로 불필요.

**구글 드라이브 쿼터 주의.** GVHMR 공개 폴더의 파일은 "Too many users have viewed or downloaded
this file recently"로 자주 막힌다(24시간). 막히면 브라우저로 그 파일을 열어 **사본 만들기** —
사본은 내 파일이라 쿼터가 적용되지 않는다. hmr2 체크포인트는 이미 그렇게 만들어둔 것을 셀 8에서 쓴다.

In [ ]:
# 6. 내 드라이브 자산 폴더에서 받기 (body_models 6개 + footmr + vitpose-wholebody)
%cd /content/FootMR
!pip install -q gdown
!gdown {ASSETS_FOLDER_ID} -O /tmp/mine

import os, shutil
!mkdir -p inputs/checkpoints/body_models inputs/checkpoints/footmr inputs/checkpoints/vitpose inputs/checkpoints/hmr2 inputs/checkpoints/yolo
!cp -r /tmp/mine/body_models/smpl /tmp/mine/body_models/smplx inputs/checkpoints/body_models/
!cp /tmp/mine/input/checkpoint/footmr/footmr_checkpoint.ckpt inputs/checkpoints/footmr/
!mv /tmp/mine/input/checkpoint/vitpose/vitpose-h-wholebody.pth inputs/checkpoints/vitpose/
!rm -f inputs/checkpoints/body_models/.DS_Store

In [ ]:
# 7. 구글을 안 거치는 경로 — yolov8x.pt는 Ultralytics 공식 릴리스에 그대로 있다.
%cd /content/FootMR
!curl -L https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8x.pt \
     -o inputs/checkpoints/yolo/yolov8x.pt

# 셀 6이 실패했을 때의 대안: FootMR Nextcloud는 공개 WebDAV를 열어두고 있어 공유 토큰을
# 사용자명으로 주면 curl로 바로 받힌다. (index.php의 `?path=&files=` 형태는 파일이 루트가 아니라
# checkpoints/ 하위에 있어서 404가 난다 — WebDAV 경로를 쓸 것.)
# !curl -u "tpLX3F6Mz4FqHaD:" "https://cloud.tnt.uni-hannover.de/public.php/webdav/checkpoints/footmr/footmr_checkpoint.ckpt" -o inputs/checkpoints/footmr/footmr_checkpoint.ckpt
# !curl -u "tpLX3F6Mz4FqHaD:" "https://cloud.tnt.uni-hannover.de/public.php/webdav/checkpoints/vitpose/vitpose-h-wholebody.pth" -o inputs/checkpoints/vitpose/vitpose-h-wholebody.pth

In [ ]:
# 8. hmr2 체크포인트 (2.6GB). 파일명이 정확히 `epoch=10-step=25000.ckpt`여야 한다.
%cd /content/FootMR
!gdown {HMR2_FILE_ID} -O "inputs/checkpoints/hmr2/epoch=10-step=25000.ckpt"

In [ ]:
# 9. 프리플라이트 — demo는 전처리를 한참 돌린 뒤에야 파일 없음으로 죽는다. 여기서 먼저 막는다.
%cd /content/FootMR
import os

NEEDED = {
    'inputs/checkpoints/hmr2/epoch=10-step=25000.ckpt': 2_709_494_041,
    'inputs/checkpoints/vitpose/vitpose-h-wholebody.pth': 2_549_087_447,
    'inputs/checkpoints/footmr/footmr_checkpoint.ckpt': 178_872_284,
    'inputs/checkpoints/yolo/yolov8x.pt': 136_890_692,
    'inputs/checkpoints/body_models/smpl/SMPL_NEUTRAL.pkl': None,
    'inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz': None,
}
bad = []
for path, want in NEEDED.items():
    size = os.path.getsize(path) if os.path.isfile(path) else 0
    ok = size > 1_000_000 and (want is None or size == want)
    # 크기까지 보는 이유: 디스크가 차면 파일이 조용히 잘린 채 저장된다.
    print(f'{"OK  " if ok else "실패"} {size/1e6:>9.1f} MB  {path}' + ('' if want is None or size == want else f'  (기대 {want/1e6:.1f} MB)'))
    if not ok:
        bad.append(path)

!df -h /content | tail -1
assert not bad, f'문제 있는 체크포인트: {bad}'
print('\n프리플라이트 통과')

In [ ]:
# 10. 입력 영상 업로드.
# 파일명에 (), @, 공백이 있으면 셸 파싱도 Hydra config override 파싱도 깨진다(2026-07-23 둘 다 실제로 걸림).
# 그래서 업로드 직후 안전한 이름으로 통일한다.
%cd /content/FootMR
from google.colab import files
import os

uploaded = files.upload()
raw_name = list(uploaded.keys())[0]
video_filename = "input_video.mp4"
os.replace(raw_name, video_filename)
print(f'{raw_name} → {video_filename}')

In [ ]:
# 11. 실행. -s = 정적 카메라(SLAM 생략, 고정 카메라 촬영이라 맞음).
# --no_postproc: 기본 후처리가 미세한 타이밍·관절 디테일을 눌러버릴 수 있다(demo.py 자체 문서화된 동작).
# 쌤쌤 코어가 "그 사람 특유의 디테일"을 보존해야 하므로 끄는 쪽이 안전하다.
# 첫 클립은 후처리 켠 것도 한 번 돌려 비교본을 남겨두면 나중에 판단 근거가 된다.
%cd /content/FootMR
!python3.10 tools/demo.py --video "{video_filename}" -s --no_postproc

## 결과

- `outputs/demo/{영상이름}/hmr4d_results.pt` — 우리가 쓸 SMPL 파라미터(`smpl_params_global`이
  world-grounded). 이걸 로컬로 내려받아 다음 두 스크립트에 넣으면 Motion Puzzle 입력 BVH가 나온다:

  ```bash
  conda run -n motion_puzzle python hmr4d_to_npz.py --pt hmr4d_results.pt --out smpl_pose.npz --fps <원본 fps>
  conda run -n motion_puzzle python retarget_smpl_to_cmu.py --npz smpl_pose.npz --out out.bvh
  ```
  `--fps`는 필수 인자다(.pt에 안 들어 있음). 촬영본이 `60000/1001`이면 60이 아니라 **59.94**로 줄 것.

- 같은 폴더의 `1_incam.mp4`(카메라 시점) / `2_global.mp4`(월드 시점)는 육안 검수용.

In [ ]:
# 12. 결과를 Drive에 백업 — Colab 세션이 끊기면 로컬 파일은 날아간다.
%cd /content/FootMR
import shutil, glob, os
for d in glob.glob('outputs/demo/*'):
    dst = f"/content/drive/MyDrive/samsam_footmr_outputs/{os.path.basename(d)}"
    shutil.copytree(d, dst, dirs_exist_ok=True)
    print('백업:', dst)